# 02 · Data Collection with Web Scraping

Parse the archived Wikipedia revision used for the capstone web-scraping lab. This table includes Falcon 9 and Falcon Heavy; it is retained as a separate context dataset and is not merged with model labels.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
html_path = ROOT / "data" / "wiki_falcon_family_2021_snapshot.html"
tables = pd.read_html(html_path)
launch_tables = [t for t in tables if "Date and time (UTC)" in t.columns]
len(launch_tables), [t.shape for t in launch_tables]

In [ ]:
frames=[]
for i, table in enumerate(launch_tables):
    t=table.copy(); t["source_table"]=i
    version_cols=[c for c in t.columns if str(c).startswith("Version, Booster")]
    if version_cols: t["Version / booster"]=t[version_cols].bfill(axis=1).iloc[:,0]
    frames.append(t)
wiki=pd.concat(frames, ignore_index=True)
date=wiki["Date and time (UTC)"].astype(str).str.extract(r"^(\d{1,2} [A-Z][a-z]+ \d{4})", expand=False)
wiki["date"]=pd.to_datetime(date, format="%d %B %Y", errors="coerce")
subset=wiki[wiki.date.between("2010-01-01","2020-12-31")].copy()
print("dated rows, 2010-2020:",len(subset))
subset[[c for c in ["Flight No.","date","Version / booster","Launch site","Orbit","Launch outcome","Booster landing"] if c in subset.columns]].head()

Source revision: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922